# 03 - 정확도/처리량 Pareto 분석

**학습 목표**: DODO 논문 Table 3의 ablation 값을 조건별로 보존하고, NED는 낮고 TPS는 높은 Pareto 후보를 계산합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 기능만 사용합니다.

표를 다시 학습하거나 재측정하는 것이 아니라 공개 수치를 산술 검산합니다.

In [ ]:
# 각 조건을 dict로 보존해 cache·attention이 다른 측정값을 한 열로 뭉개지 않습니다.
rows = [
    {'cache': 'none', 'attention': 'bidir', 'block': 32, 'ned': 0.067, 'tps': 29.5},
    {'cache': 'none', 'attention': 'bidir', 'block': 128, 'ned': 0.068, 'tps': 43.1},
    {'cache': 'none', 'attention': 'bidir', 'block': 256, 'ned': 0.057, 'tps': 42.8},
    {'cache': 'none', 'attention': 'bidir', 'block': 512, 'ned': 0.111, 'tps': 35.4},
    {'cache': 'exact', 'attention': 'block-causal', 'block': 32, 'ned': 0.069, 'tps': 103.7},
    {'cache': 'exact', 'attention': 'block-causal', 'block': 128, 'ned': 0.089, 'tps': 123.1},
    {'cache': 'exact', 'attention': 'block-causal', 'block': 256, 'ned': 0.177, 'tps': 153.4},
]

def dominates(a, b):
    no_worse = a['ned'] <= b['ned'] and a['tps'] >= b['tps']
    strictly_better = a['ned'] < b['ned'] or a['tps'] > b['tps']
    return no_worse and strictly_better

frontier = [row for row in rows if not any(dominates(other, row) for other in rows)]
for row in sorted(frontier, key=lambda x: x['ned']):
    print(row)

paper_speedup = 103.69 / 21.00
print('DODO exact-cache / AR TPS =', round(paper_speedup, 2), 'x')
assert round(paper_speedup, 1) == 4.9


## 해석

한 행이 모든 목적에서 최고이지 않습니다. 양방향 256은 더 정확하지만, exact-cache 32는 훨씬 빠릅니다. 논문의 '5×'는 반올림된 표현이며 계산값은 약 4.94×입니다. cache/attention 조건을 숨긴 단일 순위는 잘못된 결론을 만들 수 있습니다.